# Multi-Agent RAG: Choose the Smallest Coordination Topology

| Field | Value |
|---|---|
| Stage | Multi-agent RAG |
| Difficulty | Advanced |
| Status | Complete |
| Requires network/API | No |
| Last reviewed | 2026-09-25 |

Callout - Key idea:
More agents are justified only when specialization or ownership boundaries outweigh coordination cost.

## 30-Second Summary

This compatibility overview compares network, supervisor, and hierarchical-team topologies on the same two-part policy task. Three focused companion notebooks implement each topology in isolation.

## Why This Matters

A monolithic multi-agent demo hides who owns routing, evidence, verification, and stopping. A topology decision should be explicit and testable.

## Scope

| Covers | Does not cover |
|---|---|
| Topology selection, ownership, message budget, companion lessons | Live model collaboration, distributed runtime, emergent agent behavior |


## Mental Model

```text
network: peers hand off
supervisor: one router delegates
hierarchy: root -> team lead -> specialist
```


In [1]:
task = {"needs": ["policy_lookup", "arithmetic"], "teams": 1, "stable_routing": True}
topologies = {
    "network": {"best_for": "peer expertise and flexible handoffs", "base_messages": 4},
    "supervisor": {"best_for": "stable centralized routing", "base_messages": 3},
    "hierarchy": {"best_for": "multiple teams and delegated ownership", "base_messages": 5},
}


## How It Works

Selection uses workload shape, not novelty. Network peers coordinate directly, a supervisor owns routing and synthesis, and a hierarchy adds team leads only when organizational boundaries need them.


## Baseline

Choosing a hierarchy for a single stable team adds two coordination messages without improving task coverage.


In [2]:
baseline_choice = "hierarchy"
baseline = {"choice": baseline_choice, "coverage": 1.0, "messages": topologies[baseline_choice]["base_messages"]}
baseline


{'choice': 'hierarchy', 'coverage': 1.0, 'messages': 5}

## Technique Implementation

A small decision rule picks a supervisor because routing is stable, the task has distinct skills, and only one team is involved.


In [3]:
def choose_topology(task: dict) -> str:
    if task["teams"] > 1:
        return "hierarchy"
    if task["stable_routing"]:
        return "supervisor"
    return "network"

choice = choose_topology(task)
decision = {"choice": choice, "coverage": 1.0, "messages": topologies[choice]["base_messages"]}
decision


{'choice': 'supervisor', 'coverage': 1.0, 'messages': 3}

## Controlled Experiment

We compare coverage and coordination messages while holding the task and successful specialist outputs constant.


In [4]:
comparison = {
    "baseline": baseline,
    "selected": decision,
    "messages_saved": baseline["messages"] - decision["messages"],
    "companions": ["1-network.ipynb", "2-supervisor.ipynb", "3-hierarchical-teams.ipynb"],
}
comparison


{'baseline': {'choice': 'hierarchy', 'coverage': 1.0, 'messages': 5},
 'selected': {'choice': 'supervisor', 'coverage': 1.0, 'messages': 3},
 'messages_saved': 2,
 'companions': ['1-network.ipynb',
  '2-supervisor.ipynb',
  '3-hierarchical-teams.ipynb']}

## Evaluation

Both topologies cover the fixture, but the supervisor uses **3 messages instead of 5**. This is a coordination-cost comparison, not a claim that supervisors are universally best.


In [5]:
assert choice == "supervisor" and comparison["messages_saved"] == 2
assert comparison["baseline"]["coverage"] == comparison["selected"]["coverage"] == 1.0
assert len(comparison["companions"]) == 3
print("Multi-agent topology checks passed.")


Multi-agent topology checks passed.


## Decision Guide

| Condition | Topology |
|---|---|
| Flexible peer handoffs | Network |
| Stable central routing | Supervisor |
| Multiple teams/ownership layers | Hierarchy |
| One capability is enough | Single agent |


## Failure Modes and Debugging

| Symptom | Cause | Fix |
|---|---|---|
| Agent ping-pong | No owner/stop rule | Handoff budget and terminal owner |
| Duplicate retrieval | Overlapping roles | Exclusive capability contracts |
| Slow simple tasks | Over-orchestration | Single-agent fast path |
| Untraceable answer | Provenance lost in messages | Typed evidence envelope |


## Production Notes

### Observability
Trace topology, sender/receiver, role, evidence IDs, message count, and terminal owner.

### Safety and Guardrails
Handoffs cannot expand tool or data permissions.

### Latency and Cost
Budget messages and parallelize only independent specialist work.


## Practice

Change `teams` to two, explain the hierarchy decision, and assign a maximum message budget.

## Recall

Toggle - Recall: When is multi-agent RAG justified?
When real specialization or ownership boundaries repay coordination cost.

Toggle - Recall: What should cross an agent boundary?
Typed task state, evidence, provenance, and status—not an unbounded transcript.

## Sources

- [LangGraph multi-agent concepts](https://docs.langchain.com/oss/python/langchain/multi-agent)
- Repository-owned synthetic topology fixture

## Review Log

| Date | Status | Confidence | Next review focus |
|---|---|---|---|
| 2026-09-25 | Complete; executed and visually reviewed | High for the explicit topology rule | Benchmark real latency and failure recovery |
